In [1]:
from google.colab import files
files.upload()  # Upload kaggle.json here

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"agentharsha007","key":"45ad6426dfbdb7f5270509c7aaacfc86"}'}

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r  ./

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat '/content/drive/MyDrive/colab_backup/contont/': No such file or directory


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d shahrukhkhan/im2latex100k
!unzip -q im2latex100k.zip -d im2latex100k

Dataset URL: https://www.kaggle.com/datasets/shahrukhkhan/im2latex100k
License(s): CC0-1.0
 90% 568M/631M [00:06<00:01, 46.2MB/s]
100% 631M/631M [00:06<00:00, 95.1MB/s]


In [6]:
!unzip -q im2latex100k.zip -d im2latex100k

In [7]:
# Load CSV
import os
import pandas as pd

train_df = pd.read_csv("/content/im2latex100k/im2latex_train.csv")
# Add full image path
train_df['image_path'] = train_df['image'].apply(
    lambda x: os.path.join("/content/im2latex100k/formula_images_processed/formula_images_processed", x)
)
# Extract features for all images in batches
train_df['image_path']=train_df['image_path'].tolist()

In [9]:
import random
random_image_paths = random.sample(train_df['image_path'].tolist(), 32)
print(random_image_paths)

['/content/im2latex100k/formula_images_processed/formula_images_processed/97d7454fdf.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/250364a4cf.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/56d204ae0e.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/53506cfb9c.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/7e8ec0ba66.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/74bba31eb7.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/5cf66ac731.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/3a1cb39d35.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/713fcf5ede.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/7d879dcf7f.png', '/content/im2latex100k/formula_images_processed/formula_images_processed/702b06cdde.png', '/content

In [10]:
# pip install transformers torchvision pillow accelerate huggingface_hub
# Ensure 'accelerate' and 'huggingface_hub' are installed for this solution.

import warnings
warnings.filterwarnings("ignore") # Suppress warnings for a clean output

import torch
import torch.nn as nn
from typing import List, Tuple, Union
from PIL import Image
from huggingface_hub import hf_hub_download

from transformers import (
    AutoImageProcessor, SwinModel, SwinConfig,
    DetrImageProcessor, DetrForObjectDetection, DetrConfig,
    CLIPVisionModelWithProjection, CLIPImageProcessor, CLIPVisionConfig
)

# -----------------------------
# Device Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Model IDs
# -----------------------------
SWIN_ID = "microsoft/swin-base-patch4-window7-224"
DETR_ID = "facebook/detr-resnet-50"
CLIP_ID = "openai/clip-vit-base-patch32"

# -----------------------------
# Fast Image Processors
# -----------------------------
swin_proc = AutoImageProcessor.from_pretrained(SWIN_ID, use_fast=True)
detr_proc = DetrImageProcessor.from_pretrained(DETR_ID, use_fast=True)
clip_proc = CLIPImageProcessor.from_pretrained(CLIP_ID, use_fast=True)

# ----------------------------------------------------------------------
# Load pretrained backbones using from_pretrained
# This is the recommended method for loading models from Hugging Face.
# ----------------------------------------------------------------------
print("\nLoading pretrained backbones using from_pretrained...")

# --- Load Swin ---
swin = SwinModel.from_pretrained(SWIN_ID).to(device).eval()
print("Swin model loaded successfully.")

# --- Load DETR ---
detr = DetrForObjectDetection.from_pretrained(DETR_ID).to(device).eval()
print("DETR model loaded successfully.")

# --- Load CLIP Vision ---
clip_vision = CLIPVisionModelWithProjection.from_pretrained(CLIP_ID).to(device).eval()
print("CLIP Vision model loaded successfully.")
print("-" * 30)

# ----------------------------------------------------------------------

CLIP_DIM = clip_vision.config.projection_dim

def load_images(imgs: List[Union[str, Image.Image]]) -> List[Image.Image]:
    out = []
    for im in imgs:
        if isinstance(im, Image.Image):
            out.append(im.convert("RGB"))
        else:
            try:
                out.append(Image.open(im).convert("RGB"))
            except FileNotFoundError:
                print(f"Warning: Image not found at {im}. Skipping.")
    return out

class Box2DPositionalEncoding(nn.Module):
    def __init__(self, dim: int = 128, base: float = 10000.0):
        super().__init__()
        assert dim % 8 == 0, "dim must be divisible by 8"
        self.dim = dim
        self.base = base

    def forward(self, boxes: torch.Tensor) -> torch.Tensor:
        B, Q, C4 = boxes.shape
        assert C4 == 4
        d_per_coord = self.dim // 4
        freqs = d_per_coord // 2
        idx = torch.arange(freqs, device=boxes.device, dtype=boxes.dtype)
        div = torch.pow(self.base, idx / freqs).view(1, 1, 1, freqs)
        ang = boxes.unsqueeze(-1) / div
        pe_coord = torch.stack([torch.sin(ang), torch.cos(ang)], dim=-1).reshape(B, Q, 4, -1)
        return pe_coord.reshape(B, Q, self.dim)

class SwinDetrClipFeatureEncoder(nn.Module):
    def __init__(self,
                 proj_dim: int = 256,                 # CHANGED: Default projection is now 256 (DETR's dimension)
                 box_pe_dim: int = 128,
                 det_conf_thresh: float = 0.30,
                 det_max_objs: int = 30,                  # CHANGED: Reduced default object tokens for a smaller sequence
                 include_clip_token: bool = True,
                 include_detr_encoder_tokens: bool = False): # NEW: Flag to optionally exclude the largest set of tokens
        super().__init__()
        self.include_clip = include_clip_token
        self.det_conf = det_conf_thresh
        self.det_max_objs = det_max_objs
        self.include_detr_encoder_tokens = include_detr_encoder_tokens

        self.swin_proj = nn.Linear(swin.config.hidden_size, proj_dim, bias=False)
        self.detr_enc_proj = nn.Linear(detr.config.d_model, proj_dim, bias=False)
        self.detr_obj_proj = nn.Linear(detr.config.d_model, proj_dim, bias=False)
        if self.include_clip:
            self.clip_proj = nn.Linear(CLIP_DIM, proj_dim, bias=False)
        self.box_pe = Box2DPositionalEncoding(box_pe_dim)
        self.box_proj = nn.Linear(box_pe_dim, proj_dim, bias=False)
        self.type_embed = nn.Embedding(4, proj_dim) # Max 4 token types

    def forward(self, images: List[Union[str, Image.Image]]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        imgs = load_images(images)
        if not imgs: return torch.empty(0), torch.empty(0), torch.empty(0)
        B = len(imgs)

        # Use no_grad for backbone inference to save memory and compute,
        # as we only need gradients for the projection layers.
        with torch.no_grad():
            sv = swin_proc(images=imgs, return_tensors="pt")["pixel_values"].to(device)
            swin_features = swin(sv, return_dict=True).last_hidden_state

            detr_inputs = detr_proc(images=imgs, return_tensors="pt")
            dv = detr_inputs["pixel_values"].to(device)
            dm = detr_inputs.get("pixel_mask", torch.ones(dv.shape[:-1], device=device)).to(device)
            d_out = detr(dv, pixel_mask=dm, output_hidden_states=True, return_dict=True)

        # --- Token Assembly ---
        parts = []
        pad_parts = []
        type_cols = []
        type_id_counter = 0

        # 1. Swin Tokens (local features)
        swin_tok = self.swin_proj(swin_features)
        parts.append(swin_tok)
        pad_parts.append(torch.zeros((B, swin_tok.size(1)), dtype=torch.bool, device=device))
        type_cols.append(torch.full((swin_tok.size(1),), type_id_counter, dtype=torch.long, device=device))
        type_id_counter += 1

        # 2. DETR Encoder Tokens (optional dense features)
        if self.include_detr_encoder_tokens:
            detr_enc = self.detr_enc_proj(d_out.encoder_last_hidden_state)
            parts.append(detr_enc)
            pad_parts.append(torch.zeros((B, detr_enc.size(1)), dtype=torch.bool, device=device))
            type_cols.append(torch.full((detr_enc.size(1),), type_id_counter, dtype=torch.long, device=device))
            type_id_counter += 1

        # 3. DETR Object Tokens (object-centric features)
        detr_obj_features = self.detr_obj_proj(d_out.last_hidden_state)
        boxes_all = d_out.pred_boxes
        scores_all = d_out.logits.softmax(-1)[..., :-1].max(-1).values

        # CHANGED: Reduce sequence from 100 to det_max_objs by gathering top-k
        topk = torch.topk(scores_all, k=self.det_max_objs, dim=1)
        keep_indices = topk.indices.unsqueeze(-1)

        detr_obj = torch.gather(detr_obj_features, 1, keep_indices.expand(-1, -1, detr_obj_features.shape[-1]))
        boxes = torch.gather(boxes_all, 1, keep_indices.expand(-1, -1, 4))
        scores = torch.gather(scores_all, 1, topk.indices)

        keep_mask = scores >= self.det_conf
        box_pe = self.box_proj(self.box_pe(boxes))
        detr_obj = (detr_obj + box_pe).masked_fill(~keep_mask.unsqueeze(-1), 0.0)

        # CHANGED: Explicitly check if detr_obj is not empty before appending and adding type_cols
        if detr_obj.size(1) > 0:
            pad_o = ~keep_mask
            parts.append(detr_obj)
            pad_parts.append(pad_o)
            type_cols.append(torch.full((detr_obj.size(1),), type_id_counter, dtype=torch.long, device=device))
            type_id_counter += 1 # Increment type_id_counter only if detr_obj is added

        # 4. CLIP Token (optional global feature)
        if self.include_clip:
            with torch.no_grad():
                cpv = clip_proc(images=imgs, return_tensors="pt")["pixel_values"].to(device)
                clip_embeds = clip_vision(pixel_values=cpv, return_dict=True).image_embeds
            clip_tok = self.clip_proj(clip_embeds).unsqueeze(1)
            parts.append(clip_tok)
            pad_parts.append(torch.zeros((B, 1), dtype=torch.bool, device=device))
            type_cols.append(torch.full((1,), type_id_counter, dtype=torch.long, device=device))
            type_id_counter += 1 # Increment for CLIP token if included


        # Ensure type_cols is not empty before concatenating
        if not type_cols:
             # This case should ideally not happen if Swin or Clip are included,
             # but as a safeguard:
             return torch.empty(B, 0, tokens.size(-1), device=device), \
                    torch.empty(B, 0, dtype=torch.bool, device=device), \
                    torch.empty(B, 0, dtype=torch.long, device=device)


        tokens = torch.cat(parts, dim=1)
        type_ids = torch.cat(type_cols, dim=0).unsqueeze(0).expand(B, -1)
        tokens = tokens + self.type_embed(type_ids)
        pad_mask = torch.cat(pad_parts, dim=1)
        return tokens, pad_mask, type_ids

Using device: cuda


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

...

DETR model loaded successfully.


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP Vision model loaded successfully.
------------------------------


In [18]:
import random
encoder = SwinDetrClipFeatureEncoder().to(device)

# 3. Set the mode (important for training vs. inference)
encoder.train() # Use this mode if you plan to train the projection layers

# 4. Call the instance directly with your list of paths
# This will return the tokens, padding mask, and type IDs for your batch

# Print the shape to verify
# The first dimension will be the number of images in your batch (e.g., 3)
# print(f"Output tokens shape: {tokens.shape}")
# random_image_paths = random.sample(train_df['image_path'].tolist(), 32)
# tokens, pad_mask, type_ids = encoder(random_image_paths)

SwinDetrClipFeatureEncoder(
  (swin_proj): Linear(in_features=1024, out_features=256, bias=False)
  (detr_enc_proj): Linear(in_features=256, out_features=256, bias=False)
  (detr_obj_proj): Linear(in_features=256, out_features=256, bias=False)
  (clip_proj): Linear(in_features=512, out_features=256, bias=False)
  (box_pe): Box2DPositionalEncoding()
  (box_proj): Linear(in_features=128, out_features=256, bias=False)
  (type_embed): Embedding(4, 256)
)

In [11]:
import torch
import ast # Import the ast module
import os # Import os for path joining

def get_batch_data(Train_df, tree_embed_module, batch_size=32, device=None):
    """
    Prepares a batch of data for training.
    Returns:
      token_indices: [B, T]
      combined_embeds: [B, T, D]
      pad_mask: [B, T] (True = pad)
      image_paths: list[str] length B
    """
    # Resolve device inside function (avoid default-arg device pitfalls)
    if device is None:
        device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

    # Sample one set of indices and convert to Python ints for iloc
    batch_indexes = torch.randint(0, len(Train_df), (batch_size,)).tolist()

    # Collect image paths aligned with the same batch indices
    # MODIFIED: Use 'image' column from new_df as it contains the relative path
    image_paths = [Train_df.iloc[idx]["image"] for idx in batch_indexes]

    # Tokens with [START]
    batch_tokens = [['[START]'] + ast.literal_eval(Train_df.iloc[idx]["tokens"]) for idx in batch_indexes]
    print(batch_tokens[0])
    # Depths/siblings as int lists and align with [START] by prepending 0
    # MODIFIED: Use ast.literal_eval to safely parse the string representation of lists
    try:
        batch_depths = [[0] + ast.literal_eval(Train_df.iloc[idx]["depths"]) for idx in batch_indexes]
        batch_siblings = [[0] + ast.literal_eval(Train_df.iloc[idx]["siblings"]) for idx in batch_indexes]
    except (ValueError, SyntaxError) as e:
        print(f"Error parsing depths or siblings string: {e}")
        print(f"Problematic row index: {batch_indexes[0]}") # Print the index of the first problematic row
        print(f"Problematic depths string: {Train_df.iloc[batch_indexes[0]]['depths']}")
        print(f"Problematic siblings string: {Train_df.iloc[batch_indexes[0]]['siblings']}")
        raise # Re-raise the exception after printing debug info


    # Ensure all three sequences have same length per sample (truncate to min if needed)
    triplets = []
    for tokens, depths, siblings in zip(batch_tokens, batch_depths, batch_siblings):
        L = min(len(tokens), len(depths), len(siblings))
        triplets.append((tokens[:L], depths[:L], siblings[:L]))

    # Compute max length after alignment
    max_len = max(len(toks) for toks, _, _ in triplets)

    # Choose pad token from module if present, else fallback to "[PAD]"
    pad_tok = getattr(tree_embed_module, "pad_token", "[PAD]")

    # Pad and build pad mask (True where padded)
    padded_tokens, padded_depths, padded_siblings, pad_masks = [], [], [], []
    for tokens, depths, siblings in triplets:
        pad_len = max_len - len(tokens)
        padded_tokens.append(tokens + [pad_tok] * pad_len)
        padded_depths.append(depths + [0] * pad_len)
        padded_siblings.append(siblings + [0] * pad_len)
        pad_masks.append([False] * len(tokens) + [True] * pad_len)

    pad_mask = torch.tensor(pad_masks, device=device, dtype=torch.bool)

    # Map tokens to indices with robust UNK handling
    t2i = tree_embed_module.token_to_idx
    unk_idx = getattr(tree_embed_module, "unk_idx", t2i.get("[UNK]", 0))
    token_indices = torch.tensor(
        [[t2i.get(tok, unk_idx) for tok in row] for row in padded_tokens],
        dtype=torch.long # Keep on CPU initially
    )
    # Depth/sibling indices (long dtype)
    depth_indices = torch.tensor(padded_depths, dtype=torch.long) # Keep on CPU initially
    sibling_indices = torch.tensor(padded_siblings, dtype=torch.long) # Keep on CPU initially

    # MODIFIED: Move indices to the specified device before embedding lookup
    token_indices = token_indices.to(device)
    depth_indices = depth_indices.to(device)
    sibling_indices = sibling_indices.to(device)
    # Optional safety: clamp to embedding vocab ranges if available
    if hasattr(tree_embed_module, "depth_embed"):
        depth_indices.clamp_(0, tree_embed_module.depth_embed.num_embeddings - 1)
    if hasattr(tree_embed_module, "sibling_embed"):
        sibling_indices.clamp_(0, tree_embed_module.sibling_embed.num_embeddings - 1)

    # Embedding lookups and sum
    token_embeds = tree_embed_module.token_embed(token_indices)
    depth_embeds = tree_embed_module.depth_embed(depth_indices)
    sibling_embeds = tree_embed_module.sibling_embed(sibling_indices)
    combined_embeds = token_embeds + depth_embeds + sibling_embeds

    # MODIFIED: Construct full image paths using the base directory
    # image_paths_full = [os.path.join("/content/im2latex100k/formula_images_processed/formula_images_processed", img_name) for img_name in image_paths]

    decoder_input_indices = token_indices[:, :-1]
    decoder_input_embeds = combined_embeds[:, :-1, :]
    decoder_input_mask = pad_mask[:, :-1]
    decoder_target_indices = token_indices[:, 1:]
    return decoder_input_indices,decoder_target_indices,decoder_input_mask,decoder_input_embeds,image_paths
def get_vocabulary(vocab_filepath, add_specials=True):
    with open(vocab_filepath, "r", encoding="utf-8") as f:
        vocab = list(filter(None, (line.strip() for line in f)))

    if add_specials:
        for tok in ["[PAD]", "[UNK]", "[START]", "[END]"]:
            if tok not in vocab:
                vocab.append(tok)
    vocab.append(" ")
    return vocab


In [12]:
class TreeEmbedding(nn.Module):
    def __init__(self, vocab_list, embed_dim=256, max_depth=20, max_sibling=225):
        super().__init__()
        self.pad_token_id=vocab_list.index("[PAD]")
        self.token_to_idx = {tok: i for i, tok in enumerate(vocab_list)}
        self.token_embed = nn.Embedding(len(vocab_list), embed_dim,padding_idx=self.pad_token_id)
        self.depth_embed = nn.Embedding(max_depth, embed_dim)
        self.sibling_embed = nn.Embedding(max_sibling, embed_dim)
        self.embed_dim = embed_dim

In [13]:
import pandas as pd, csv
new_df = pd.read_csv(
    "./Latex_img_node.csv",
    engine="python",           # more tolerant parser
    on_bad_lines="skip"        # or "warn" to see which lines were skipped
)
print(new_df['image'].sample(n=5))

56814    /content/im2latex100k/formula_images_processed...
47178    /content/im2latex100k/formula_images_processed...
26706    /content/im2latex100k/formula_images_processed...
62801    /content/im2latex100k/formula_images_processed...
6041     /content/im2latex100k/formula_images_processed...
Name: image, dtype: object


In [19]:
import pandas as pd, csv
import torch # Import torch to get device
#test tree working
vocab_list=get_vocabulary("/content/vocab.txt")
tree=TreeEmbedding(vocab_list)
print("Vocabulary size:", len(vocab_list))
print(vocab_list[:20])  # print first 20 tokens
print("[PAD] index:", vocab_list.index("[PAD]"))
print("[UNK] index:", vocab_list.index("[UNK]"))
print("[START] index:", vocab_list.index("[START]"))
# MODIFIED: Move the tree embedding module to the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tree.to(device)

# Assuming new_df is loaded from the CSV
decoder_input,decoder_target,embeds,pad_masks,img_paths=get_batch_data(new_df,tree, device=device)
print(decoder_input[0])
print(decoder_target[0])
print(embeds[0])
print(pad_masks[0])
# print(target_tokens[9])
print("Successfully generated batch data.")
print(f"Token indices shape: {decoder_input.shape}")
print(f"Combined embeds shape: {embeds.shape}")
print(f"Pad mask shape: {pad_masks.shape}")
print(f"Number of image paths: {len(img_paths)}")
tokens, pad_mask, type_ids =encoder(img_paths)
print(tokens.shape)
print(pad_mask[0])

# print(target_tokens.shape)

Vocabulary size: 805
['[START]', '[PAD]', '[UNK]', '[/GROUP]', '[GROUP]', '_', '^', '[/ARG]', '[ARG]', '(', ')', '2', '=', '1', '-', ',', '[/FRAC]', '[FRAC]', '+', 'i']
[PAD] index: 1
[UNK] index: 2
[START] index: 0
['[START]', '[NABLA]', '[/NABLA]', '_', ' ', '[GROUP]', ' ', 'X', ' ', '_', ' ', '[GROUP]', ' ', 'f', ' ', '[/GROUP]', ' ', '[/GROUP]', ' ', '=', ' ', 'X', ' ', '_', ' ', '[GROUP]', ' ', 'f', ' ', '[/GROUP]', ' ', '-', ' ', 'i', ' ', '[VARTHETA]', '[/VARTHETA]', '(', ' ', 'X', ' ', '_', ' ', '[GROUP]', ' ', 'f', ' ', '[/GROUP]', ' ', ')', ' ', '[;]', '[/;]', ' ', ',']
tensor([  0, 196, 195,   5, 804,   4, 804, 132, 804,   5, 804,   4, 804,  82,
        804,   3, 804,   3, 804,  12, 804, 132, 804,   5, 804,   4, 804,  82,
        804,   3, 804,  14, 804,  19, 804, 282, 281,   9, 804, 132, 804,   5,
        804,   4, 804,  82, 804,   3, 804,  10, 804,  48,  47, 804,  15,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1, 

In [20]:
def get_training_data(Train_df,image_encoder,Tree,batch_size=32,device=None):
  if device is None:
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
  decoder_input,decoder_output,pad_masks,decoder_embed,img_paths=get_batch_data(Train_df,Tree,batch_size,device)
  image_features,image_masks=image_encoder(img_paths)
  return ((image_features,image_masks),(decoder_embed,pad_masks)),decoder_output

In [21]:
import math
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, embed_dim]
        self.register_buffer('pe', pe)
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
class RandomFourier2D(torch.nn.Module):
    def __init__(self, out_dim: int, sigma: float = 10.0, seed: int | None = None):
        super().__init__()
        assert out_dim % 2 == 0
        k = out_dim // 2
        g = torch.Generator()
        if seed is not None: g.manual_seed(seed)
        B = torch.randn(2, k, generator=g) * sigma
        self.register_buffer("B", B)
    def forward(self, pos):  # pos: (N,2) in [0,1] or pixel coords normalized
        proj = pos @ self.B                   # [N, k]
        return torch.cat([torch.sin(2*math.pi*proj),
                          torch.cos(2*math.pi*proj)], dim=-1)  # [N, out_dim]
class encoderblock(torch.nn.Module):
    def __init__(self,num_heads, embed_size, dropout,forward_expansion):
      super().__init__()
      self.attention=nn.MultiheadAttention(embed_size,num_heads=num_heads)
      self.feedforward=nn.Sequential(
          nn.Linear(embed_size,forward_expansion*embed_size),
          nn.ReLU(),
          nn.Linear(forward_expansion*embed_size,embed_size)
      )
      self.norm1=nn.LayerNorm(embed_size)
      self.norm2=nn.LayerNorm(embed_size)
      self.dropout1=nn.Dropout(dropout)
      self.dropout2=nn.Dropout(dropout)
    def forward(self,input,padding_mask=None):
      x=input.transpose(0,1)
      if padding_mask is not None:
          attention_out,_=self.attention(x,x,x,key_padding_mask=padding_mask)
      else:
          attention_out,_=self.attention(x,x,x)
      add_norm=self.norm1(self.dropout1(attention_out)+x)
      x=self.feedforward(add_norm)
      x=self.norm2(add_norm+self.dropout2(x))
      x=x.transpose(0,1)
      return x
class decoderblock(torch.nn.Module):
  def __init__(self,num_heads,embed_size,dropout,forward_expasion):
    super().__init__()
    self.cross_attention=nn.MultiheadAttention(embed_size,num_heads=num_heads)
    self.attention=nn.MultiheadAttention(embed_size,num_heads=num_heads)
    self.dropout1=nn.Dropout(dropout)
    self.dropout2=nn.Dropout(dropout)
    self.dropout3=nn.Dropout(dropout)
    self.feed_forward=nn.Sequential(
        nn.Linear(embed_size,forward_expasion*embed_size),
        nn.ReLU(),
        nn.Linear(forward_expasion*embed_size,embed_size)
    )
    self.norm1=nn.LayerNorm(embed_size)
    self.norm2=nn.LayerNorm(embed_size)
    self.norm3=nn.LayerNorm(embed_size)
  def forward(self,encoder_input,decoder_input,padding_mask=None,attn_mask=None):
    x=decoder_input.transpose(0,1)
    attention_out,_=self.attention(x,x,x,attn_mask=attn_mask,key_padding_mask=padding_mask)
    add_norm=self.norm1(self.dropout1(attention_out)+x)
    cross_attention_out,_=self.cross_attention(add_norm,encoder_input.transpose(0,1),encoder_input.transpose(0,1))
    add_norm=self.norm2(self.dropout2(cross_attention_out)+add_norm)
    feed_out=self.feed_forward(add_norm)
    x=self.norm3(self.dropout3(feed_out)+add_norm)
    x=x.transpose(0,1)
    return x

In [27]:
class ImageToLatexTransformer(nn.Module):
    def __init__(self,num_blocks=6,num_heads=8,vocabulary=None, vocab_size=None, embed_size=256,cnn_input_size=256, dropout=0.1,forward_expansion=4):
        super(ImageToLatexTransformer, self).__init__()
        self.device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.pad_token_id=vocab_list.index("[PAD]")
        self.project_cnn=nn.Linear(cnn_input_size,embed_size)
        # self.positional_encoding_fourier=RandomFourier2D(embed_size)
        # self.positional_encoding_encoder=SinusoidalPositionalEncoding(embed_size)
        self.positional_encoding_decoder=SinusoidalPositionalEncoding(embed_size)
        self.TreeEmbedder = TreeEmbedding(vocabulary,embed_size, max_depth=20, max_sibling=225)
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=self.pad_token_id)
        self.encoder=nn.ModuleList([encoderblock(num_heads,embed_size,dropout,forward_expansion) for _ in range(num_blocks)])
        self.decoder=nn.ModuleList([decoderblock(num_heads,embed_size,dropout,forward_expansion) for _ in range(num_blocks)])
        self.final_layer=nn.Linear(embed_size,vocab_size)
        self.value_head = nn.Linear(embed_size, 1)
    def forward(self,input,return_hidden=False):
        ((cnn_features,cnn_masks),(input_seq,input_masks))=input
        cnn_features=self.project_cnn(cnn_features)
        # cnn_features=self.positional_encoding_encoder(cnn_features)
        for layer in self.encoder:
            cnn_features=layer(cnn_features,cnn_masks)
        #process text_features
        B,T=input_seq.shape
        attention_mask=torch.triu(torch.ones(T,T, device=self.device) * float('-inf'), diagonal=1)
        # input_seq=self.embedding(input_seq)
        input_seq=self.positional_encoding_decoder(input_seq)
        for layer in self.decoder:
            input_seq=layer(cnn_features,input_seq,input_masks,attention_mask)
        logits=self.final_layer(input_seq)
        values = self.value_head(input_seq).squeeze(-1)
        if return_hidden:
          return logits, values
        return logits

In [29]:
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import trange

# --- Model & Optimizer Initialization ---
im2latexmodel = ImageToLatexTransformer(vocabulary=vocab_list,vocab_size=len(vocab_list))
im2latexmodel.to(im2latexmodel_opt.device)
device = im2latexmodel_opt.device
optimizer = torch.optim.Adam(
    im2latexmodel_opt.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),  # Common choice in Transformer training
    eps=1e-9
)
epochs = 10000
# --- Scheduler Initialization ---
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=epochs,      # Full cycle length in epochs
    eta_min=1e-6       # Minimum LR at the end of cosine
)
# --- Training Loop ---
pbar = trange(epochs, desc="Training", ncols=100)
for epoch in pbar:
    im2latexmodel_opt.train()
    input, output_seq = get_batch_data()
    # # Fetch batch
    # (cnn_features, input_seq), output_seq = get_batch_data(
    #     train_df, padded, batch_size=16, device=device
    # )
    logits = im2latexmodel_opt(input)  # [B, T, vocab]
    B, T, S = logits.shape
    logits = logits.view(B * T, S)
    output_seq = output_seq.view(B * T)
    optimizer.zero_grad()
    loss = torch.nn.functional.cross_entropy(
        logits, output_seq, ignore_index=vocab_list.index("[PAD]")
    )
    loss.backward()
    # Gradient clipping for stability
    # torch.nn.utils.clip_grad_norm_(im2latexmodel_opt.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()  # Update LR
    # Show LR and loss in progress bar
    current_lr = scheduler.get_last_lr()[0]
    pbar.set_description(
        f"Epoch {epoch} | Loss: {loss.item():.4f} | LR: {current_lr:.2e}"
    )
# --- Save model ---
torch.save(im2latexmodel_opt.state_dict(), "model_weights_opt.pth")

Training:   0%|                                                           | 0/10000 [00:00<?, ?it/s]


KeyError: 'tokens'